In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from astroquery.vizier import Vizier
from scope.utils import parse_load_config, write_parquet
from penquins import Kowalski


# # Get gaia white dwarf catalog
# vizier = Vizier()
# vizier.ROW_LIMIT = 10
# vizier.COL_LIMIT = 10
# wd_cat = vizier.get_catalogs('J/MNRAS/508/3877/maincat')[0].to_pandas()
from astropy.coordinates import SkyCoord
from astropy import units as u
from tqdm.auto import tqdm


In [2]:
config = parse_load_config()

timeout = config['kowalski']['timeout']
hosts = [x
         for x in config['kowalski']['hosts'] 
         if config['kowalski']['hosts'][x]['token'] is not None
]
instances = {
host: {
    'protocol': config['kowalski']['protocol'],
    'port': config['kowalski']['port'],
    'host': f'{host}.caltech.edu',
    'token': config['kowalski']['hosts'][host]['token'],
    }
    for host in hosts
}

kowalski_instances = Kowalski(timeout=timeout, instances=instances)

No --config-path specified. Loading '/home/peytonejohnsons/wdb-ml/config.yaml'.


In [3]:
from astropy.table import Table
gaia_wds = Table.read('data/gaiaedr3_wd_main.fits').to_pandas()

## Random Gaia WD candidates with non-variable SCoPe matches

Set `n_wds` to the number of random Gaia white dwarf candidates you want to keep after crossmatching to SCoPe and applying the non-variable cut.


In [4]:
n_wds = 100
seed = 123
radius_arcsec = 2.0
vnv_limit = 0.1
batch_size = 1000
max_draws = 200_000

FEATURES_CATALOG = config["kowalski"]["collections"]["features"]
CLASSIFICATIONS_CATALOG = config["kowalski"]["collections"]["classifications"]
SOURCES_CATALOG = config["kowalski"]["collections"]["sources"]


In [8]:
def _decode_value(value):
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore").strip()
    return value


def _add_gaia_columns(match, wd_row):
    gaia_cols = [
        "WDJ_name",
        "source_id",
        "ra",
        "dec",
        "Pwd",
        "phot_g_mean_mag",
        "phot_bp_mean_mag",
        "phot_rp_mean_mag",
        "bp_rp",
        "parallax",
        "parallax_error",
        "pmra",
        "pmdec",
        "absG",
    ]
    for col in gaia_cols:
        if col in wd_row.index:
            match[f"gaia_{col}"] = _decode_value(wd_row[col])
    return match


def crossmatch_random_gaia_wds_to_nonvariable_scope(
    gaia_wds,
    n_wds=100,
    seed=42,
    radius_arcsec=2.0,
    vnv_limit=0.1,
    batch_size=1000,
    max_draws=200_000,
):
    candidates = (
        gaia_wds
        .dropna(subset=["ra", "dec", "source_id"])
        .sample(n=min(max_draws, len(gaia_wds)), random_state=seed)
        .reset_index(drop=True)
    )

    matches = []

    for start in tqdm(range(0, len(candidates), batch_size)):
        batch = candidates.iloc[start:start + batch_size].copy()
        radec = {
            str(i): [float(row.ra), float(row.dec)]
            for i, row in batch.iterrows()
        }

        query = {
            "query_type": "cone_search",
            "query": {
                "object_coordinates": {
                    "cone_search_radius": radius_arcsec,
                    "cone_search_unit": "arcsec",
                    "radec": radec,
                },
                "catalogs": {
                    FEATURES_CATALOG: {
                        "projection": {
                            "_id": 1,
                            "ra": 1,
                            "dec": 1,
                            "period": 1,
                            "field": 1,
                            "ccd": 1,
                            "quad": 1,
                            "filter": 1,
                            "Gaia_EDR3___id": 1,
                            "Gaia_EDR3__phot_g_mean_mag": 1,
                            "Gaia_EDR3__phot_bp_mean_mag": 1,
                            "Gaia_EDR3__phot_rp_mean_mag": 1,
                            "Gaia_EDR3__parallax": 1,
                            "Gaia_EDR3__parallax_error": 1,
                        }
                    },
                    CLASSIFICATIONS_CATALOG: {
                        "projection": {
                            "_id": 1,
                            "vnv_dnn": 1,
                            "vnv_xgb": 1,
                            "bis_dnn": 1,
                            "bis_xgb": 1,
                            "cv_dnn": 1,
                            "cv_xgb": 1,
                        }
                    },
                },
            },
            "kwargs": {"filter_first": False},
        }

        response = kowalski_instances.query(query=query).get("gloria")
        if response.get("status") != "success":
            print(response.get("message"))
            continue

        feature_data = response["data"][FEATURES_CATALOG]
        classification_data = response["data"][CLASSIFICATIONS_CATALOG]

        for key in radec:
            wd_row = batch.loc[int(key)]
            df_f = pd.DataFrame(feature_data.get(key, []))
            df_c = pd.DataFrame(classification_data.get(key, []))
            if df_f.empty or df_c.empty:
                continue

            df = df_f.merge(df_c, on="_id", how="inner")
            if df.empty or not {"vnv_dnn", "vnv_xgb"}.issubset(df.columns):
                continue

            df = df[(df["vnv_dnn"] < vnv_limit) & (df["vnv_xgb"] < vnv_limit)]
            if df.empty:
                continue

            wd_coord = SkyCoord(wd_row.ra * u.deg, wd_row.dec * u.deg)
            scope_coord = SkyCoord(df["ra"].values * u.deg, df["dec"].values * u.deg)
            df["sep_arcsec"] = wd_coord.separation(scope_coord).arcsec

            best = df.sort_values("sep_arcsec").iloc[0].to_dict()
            matches.append(_add_gaia_columns(best, wd_row))

        if len(matches) >= n_wds:
            break

    return pd.DataFrame(matches).head(n_wds)


In [9]:
nonvariable_wd_scope = crossmatch_random_gaia_wds_to_nonvariable_scope(
    gaia_wds,
    n_wds=n_wds,
    seed=seed,
    radius_arcsec=radius_arcsec,
    vnv_limit=vnv_limit,
    batch_size=batch_size,
    max_draws=max_draws,
)

print(f"Found {len(nonvariable_wd_scope)} non-variable SCoPe matches")
nonvariable_wd_scope.head()


  0%|▍                                                                                  | 1/200 [00:05<19:38,  5.92s/it]

Found 100 non-variable SCoPe matches


,_id,ra,dec,field,ccd,quad,filter,Gaia_EDR3___id,Gaia_EDR3__phot_g_mean_mag,Gaia_EDR3__phot_bp_mean_mag,...,gaia_Pwd,gaia_phot_g_mean_mag,gaia_phot_bp_mean_mag,gaia_phot_rp_mean_mag,gaia_bp_rp,gaia_parallax,gaia_parallax_error,gaia_pmra,gaia_pmdec,gaia_absG
0,1.056163e+13,78.408063,21.065998,561.0,16.0,4.0,1.0,3.414174e+18,20.207565,20.420800,...,0.971619,20.207565,20.420799,19.794540,0.626259,4.531075,0.705873,46.669054,22.475312,13.488572
1,1.063759e+13,277.454925,28.331561,637.0,15.0,4.0,1.0,4.587222e+18,20.173819,20.156120,...,0.974111,20.173819,20.156120,20.428675,-0.272554,0.875338,0.467702,2.060809,-6.067056,9.884697
2,1.048959e+13,294.423430,6.787609,489.0,15.0,4.0,2.0,4.294904e+18,20.393402,20.840393,...,0.136968,20.393402,20.840393,19.780704,1.059689,1.725181,0.938681,-1.348515,-6.740245,11.577575
3,1.063755e+13,280.266795,28.240863,637.0,14.0,4.0,1.0,4.539369e+18,19.878298,19.748014,...,0.999299,19.878298,19.748014,20.053925,-0.305910,1.700675,0.340812,-16.945655,-12.884440,11.031405
4,1.069944e+13,42.384106,42.302462,699.0,12.0,1.0,1.0,3.372231e+17,19.636417,19.650373,...,0.981661,19.636417,19.650373,19.625376,0.024998,1.937112,0.378050,11.305174,-6.306111,11.072191


In [12]:
output_path = f"data/random_{n_wds}_gaia_wd_scope_nonvariable.parquet"
write_parquet(nonvariable_wd_scope, output_path)
output_path


'data/random_100_gaia_wd_scope_nonvariable.parquet'

In [13]:
nonvariable_wd_scope.columns

Index(['_id', 'ra', 'dec', 'field', 'ccd', 'quad', 'filter', 'Gaia_EDR3___id',
       'Gaia_EDR3__phot_g_mean_mag', 'Gaia_EDR3__phot_bp_mean_mag',
       'Gaia_EDR3__phot_rp_mean_mag', 'Gaia_EDR3__parallax',
       'Gaia_EDR3__parallax_error', 'bis_dnn', 'vnv_dnn', 'cv_dnn', 'bis_xgb',
       'cv_xgb', 'vnv_xgb', 'sep_arcsec', 'gaia_WDJ_name', 'gaia_source_id',
       'gaia_ra', 'gaia_dec', 'gaia_Pwd', 'gaia_phot_g_mean_mag',
       'gaia_phot_bp_mean_mag', 'gaia_phot_rp_mean_mag', 'gaia_bp_rp',
       'gaia_parallax', 'gaia_parallax_error', 'gaia_pmra', 'gaia_pmdec',
       'gaia_absG'],
      dtype='object')